# Patent Analysis System
This notebook processes a patent document and answers questions about it using LLaMA.

## Required packages:
```bash
pip install PyMuPDF easyocr sentence-transformers qdrant-client tqdm
```

## Required files:
1. US11960514.pdf - The patent document
2. questions.txt - List of questions to answer


In [ ]:
# do not change

# Import required libraries
import fitz  # PyMuPDF
import easyocr
import os
import json
import time
import subprocess
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, PointStruct

# Initialize models
reader = easyocr.Reader(['en'])
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
qdrant = QdrantClient(":memory:")


# do not change


def extract_text_from_pdf(pdf_path):#process of extracting from pdf , whether using png or ocr
    """Extract text from PDF using OCR only when needed"""
    doc = fitz.open(pdf_path)
    text_chunks = []

    for page_num in tqdm(range(len(doc)), desc="Extracting text from PDF"):
        page = doc.load_page(page_num)
        raw_text = page.get_text().strip()

        if raw_text:
            text = clean_text(raw_text)
        else:
            # Fallback to OCR only if no text is found
            pix = page.get_pixmap(dpi=150)  # Lower DPI for speed
            img_bytes = pix.tobytes("png")
            ocr_result = reader.readtext(img_bytes, detail=0)
            text = clean_text("\n".join(ocr_result))

        if text:
            # Optional: split long text into smaller chunks
            for chunk in split_text(text, max_length=1000):
                text_chunks.append({
                    "page": page_num + 1,
                    "content": chunk
                })

    return text_chunks


def store_vectors(text_chunks):#here we create the patent content into vectors
    """Convert text to vectors and store in Qdrant"""
    qdrant.recreate_collection(
        collection_name="patent_content",
        vectors_config=VectorParams(size=384, distance=Distance.COSINE)
    )

    points = []
    for idx, chunk in enumerate(text_chunks):
        vector = embedding_model.encode(chunk["content"]).tolist()
        points.append(PointStruct(
            id=idx,
            vector=vector,
            payload=chunk
        ))

    qdrant.upsert(collection_name="patent_content", points=points)
    print(f"✓ Stored {len(points)} text chunks as vectors")


# Utility functions
import re
from textwrap import wrap

def clean_text(text):#text normalization meaning stripping from no ascii chars
    """Basic text normalization"""
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\x00-\x7F]+', '', text)  # remove non-ASCII
    return text.strip()

def split_text(text, max_length=1000):#splitting into sizes of 1000 
    """Split long text into smaller chunks"""
    return wrap(text, width=max_length)

f
# Process the patent document
pdf_path = "patent.pdf"
text_chunks = extract_text_from_pdf(pdf_path)
store_vectors(text_chunks)


# 83/100


import re
import subprocess
import time
from tqdm import tqdm

def preprocess_question(question):#here we send the text into the model as a prompt with question context and prompt
    """Use local LLM to rewrite question into focused embedding query"""

    prompt = f"""You are a semantic optimizer for technical patent questions. Rewrite the question below into a concise, embedding-friendly query that preserves only the explicit technical intent, without adding assumptions or external knowledge.

    Original Question: {question}
    Optimized Query:"""


    try:
        result = subprocess.run(
            ["ollama", "run", "llama3", prompt],
            capture_output=True,
            text=True,
            encoding='utf-8',
            errors='replace'
        )
        return result.stdout.strip()
    except Exception as e:
        print(f"Error rewriting question: {str(e)}")
        return question.lower().strip()


def clean_text(text):#clean the text out of non ascii chars
    """Clean and normalize extracted text"""
    text = re.sub(r'\\\(.*?\\\)', '', text)  # remove bracketed codes
    text = re.sub(r'\s+', ' ', text)        # normalize whitespace
    text = re.sub(r'[^\x00-\x7F]+', '', text)  # remove non-ASCII
    return text.strip()


def get_context(question, top_k=5):#/**allows from */
    """Find relevant context for a question with fallback and filtering"""
    question_clean = preprocess_question(question)#getquestion in one line
    question_vector = embedding_model.encode(question_clean).tolist()#get context in vector embedding using cosine similarity

    results = qdrant.search(#search the db qdrant for querry
        collection_name="patent_content",
        query_vector=question_vector,
        limit=top_k
    )

    contexts = []
    for hit in results:
       # /**context up to 3000 chars*/
        page = hit.payload["page"]
        text = clean_text(hit.payload["content"])
        # סינון לפי אורך ומילות מפתח מהשאלה#limit answer for not being too short
        if len(text) > 100 and any(word in text.lower() for word in question_clean.split()):
            contexts.append(f"[Page {page}]:\n{text}")

    if not contexts:
        return "⚠️ No relevant context found in the patent document."

    context_text = "\n\n".join(contexts)#join the contexts
    return context_text[:3000]


def truncate_to_300_chars(text):
    """Truncate text to the last full sentence within 300 characters."""
    if len(text) <= 300:
        return text.strip()
    truncated = text[:300]
    match = re.search(r'(?s)^(.{0,300}?[\\.\\!\\?])\\s', truncated)
    if match:
        return match.group(1).strip()
    return truncated.strip()


def get_answer(question, context):#build the prompt as prompt+context+question
    """Get answer from LLaMA using context with improved prompt and sentence-aware truncation."""

    prompt = f"""You are a technical patent analyst. Use only the content below. Do not infer, assume, or invent anything not explicitly stated.

        Instructions:
        - Answer using only the provided excerpts.
        - Use exact terminology and module names from the patent.
        - If the question refers to a figure or flowchart, follow the numbered steps precisely.
        - Do not add explanations, definitions, or generalizations.
        - Avoid phrases like "Based on the patent" or "According to".
        - Use active voice and compact phrasing.
        - Limit the answer to 300 characters.
        - Always cite the page number(s) from which the answer was derived, using the format: (Page X).
        
        Patent Content:
        {context}
        
        Question:
        {question}
        
        Answer (max 300 characters, include source page):"""
        
            
    try:
        result = subprocess.run(
            ["ollama", "run", "llama3", prompt],
            capture_output=True,
            text=True,
            encoding='utf-8',
            errors='replace'
        )
        raw_output = result.stdout.strip()
        return truncate_to_300_chars(raw_output)
    except Exception as e:
        print(f"Error: {str(e)}")
        return "Error generating response"


# Read questions
with open("questions.txt", "r", encoding="utf-8") as f:
    questions = [line.strip() for line in f if line.strip()]

# Process questions and generate answers
answers = []
for question in tqdm(questions, desc="Answering questions"):
    context = get_context(question)
    answer = get_answer(question, context)
    answers.append(f"Q: {question}\nA: {answer}")

    # ==== DEBUG LOGGING: REMOVE AFTER DEBUGGING ====
    with open("contexts.txt", "a", encoding="utf-8") as f:
        f.write(f"\n\nQ: {question}\n{context}\n")
    # ===============================================

    # Save progress
    with open("answers.txt", "w", encoding="utf-8") as f:
        f.write("\n\n".join(answers))

    time.sleep(2)  # Rate limiting
print("✓ All answers saved to answers.txt")


Extracting text from PDF: 100%|███████████████████████████████████████████████████████| 45/45 [00:00<00:00, 108.24it/s]
C:\Users\dima1\AppData\Local\Temp\ipykernel_8256\3049988591.py:55: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant.recreate_collection(
C:\Users\dima1\anaconda3\envs\GenAI2005_CUDA\lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


✓ Stored 221 text chunks as vectors


Answering questions:   0%|                                                                      | 0/10 [00:00<?, ?it/s]

In [29]:
# 84/100

import re
import subprocess
import time
from tqdm import tqdm

def preprocess_question(question):
    """Use local LLM to rewrite question into focused embedding query"""
    prompt = f"""You are a semantic rewriter for patent-related questions. Rewrite the following question into a focused, embedding-friendly query that captures the core technical intent.

Question: {question}
Rewritten:"""

    try:
        result = subprocess.run(
            ["ollama", "run", "llama3", prompt],
            capture_output=True,
            text=True,
            encoding='utf-8',
            errors='replace'
        )
        return result.stdout.strip()
    except Exception as e:
        print(f"Error rewriting question: {str(e)}")
        return question.lower().strip()


def clean_text(text):
    """Clean and normalize extracted text"""
    text = re.sub(r'\\\(.*?\\\)', '', text)  # remove bracketed codes
    text = re.sub(r'\s+', ' ', text)        # normalize whitespace
    text = re.sub(r'[^\x00-\x7F]+', '', text)  # remove non-ASCII
    return text.strip()


def get_context(question, top_k=5):
    """Find relevant context for a question with fallback and filtering"""
    question_clean = preprocess_question(question)
    question_vector = embedding_model.encode(question_clean).tolist()

    results = qdrant.search(
        collection_name="patent_content",
        query_vector=question_vector,
        limit=top_k
    )

    contexts = []
    for hit in results:
        page = hit.payload["page"]
        text = clean_text(hit.payload["content"])
        # סינון לפי אורך ומילות מפתח מהשאלה
        if len(text) > 100 and any(word in text.lower() for word in question_clean.split()):
            contexts.append(f"[Page {page}]:\n{text}")

    if not contexts:
        return "⚠️ No relevant context found in the patent document."

    context_text = "\n\n".join(contexts)
    return context_text[:3000]



def get_answer(question, context):
    """Get answer from LLaMA using context"""
    prompt = f"""You are a patent analyst. Based only on the excerpts below, answer the question precisely and technically. Do not use external knowledge or assumptions.

Patent Content:
{context}

Question: {question}

Answer (max 300 characters):"""

    try:
        result = subprocess.run(
            ["ollama", "run", "llama3", prompt],
            capture_output=True,
            text=True,
            encoding='utf-8',
            errors='replace'
        )
        return result.stdout.strip()[:300]
    except Exception as e:
        print(f"Error: {str(e)}")
        return "Error generating response"


# Read questions
with open("questions.txt", "r", encoding="utf-8") as f:
    questions = [line.strip() for line in f if line.strip()]

# Process questions and generate answers
answers = []
for question in tqdm(questions, desc="Answering questions"):
    context = get_context(question)
    answer = get_answer(question, context)
    answers.append(f"Q: {question}\nA: {answer}")

    # ==== DEBUG LOGGING: REMOVE AFTER DEBUGGING ====
    with open("contexts.txt", "a", encoding="utf-8") as f:
        f.write(f"\n\nQ: {question}\n{context}\n")
    # ===============================================

    # Save progress
    with open("answers.txt", "w", encoding="utf-8") as f:
        f.write("\n\n".join(answers))

    time.sleep(2)  # Rate limiting
print("✓ All answers saved to answers.txt")

C:\Users\stnw24\AppData\Local\Temp\ipykernel_9212\2973862193.py:41: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = qdrant.search(
Answering questions: 100%|█████████████████████████████████████████████████████████████| 10/10 [01:14<00:00,  7.40s/it]

✓ All answers saved to answers.txt
